In [2]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import math
import time
import random
from io import StringIO
from datetime import date
import pickle

In [ ]:
base_url = "https://www.milesplit.com/results/?"
years = [str(x) for x in list(range(2006,2026))]
months = [str(x) for x in list(range(1,13))]
levels = ["hs"]

In [ ]:
links = []

for l in levels:
    for y in years:
        for m in months:
            url = base_url + "month=" + m + "&year=" + y + "&level=" + l
            print(url)
            r = requests.get(url)
            soup = BeautifulSoup(r.content, 'html.parser')
            tags = soup.select("td.name a")
            hrefs = [a['href'] for a in tags if 'href' in a.attrs]
            links.extend(hrefs)
            #time.sleep(10)

In [3]:
with open("meet_links.pkl", "rb") as f:
    tmp_links = pickle.load(f)

len(tmp_links)

7583

In [ ]:
import re
import os

directory = "data/"
files = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
print(files)

for file in files:
    with open(f"data/{file}", "rb") as f:
        tmp_links = pickle.load(f)

    for k in tmp_links:
        print(k)
        kclean = re.sub(r'[^a-zA-Z0-9-]', '_', k)
        #os.makedirs(f"data/{kclean}", exist_ok=True)
        for l in tmp_links[k]:
            lclean = re.sub(r'[^a-zA-Z0-9-]', '_', l).split(re.sub(r'[^a-zA-Z0-9-]', '_', k))[1]
            print("\t", lclean)
            #if lclean == "":
                #os.makedirs(f"data/{kclean}/default", exist_ok=True)
                # TODO NO CREATING DEFUALT
            #else:
                #os.makedirs(f"data/{kclean}/{lclean}", exist_ok=True)

In [ ]:
for l in links:
    print(f"Scraping: {l}")
    r = requests.get(l)
    soup = BeautifulSoup(r.content, 'html.parser')
    tags = soup.select("a")
    
    # Extract hrefs that contain 'raw' and start with 'https://'
    hrefs = [a['href'] for a in tags if 'href' in a.attrs]
    for h in hrefs:
        if "raw" in h and h.startswith("https://"):
            print("---", h)

In [ ]:
for l in links:
    #time.sleep(10)
    # Send a GET request to the URL
    response = requests.get(l)
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the <pre> tag that contains the raw results
    pre_tag = soup.find('pre')
    if pre_tag:
        raw_results = pre_tag.get_text()
        print("Raw results found.", l)
        #print(raw_results)
    else:
        print("No raw results found on the page.", l)

In [ ]:
import requests
from bs4 import BeautifulSoup

raw_links = dict()

for l in links:
    raw = []
    print(f"Scraping: {l}")
    try:
        r = requests.get(l)
        r.raise_for_status()
    except Exception as e:
        print(f"Failed to fetch {l}: {e}")
        continue

    soup = BeautifulSoup(r.content, 'html.parser')
    tags = soup.select("a")
    hrefs = [a['href'] for a in tags if 'href' in a.attrs]

    for h in hrefs:
        if h.startswith("https://"):
            try:
                resp = requests.get(h, allow_redirects=True)
                final_url = resp.url

                if "raw" in final_url:
                    #print("---", final_url)
                    raw.append(final_url)
                else:
                    # One level deeper: scrape this page too
                    try:
                        soup2 = BeautifulSoup(resp.content, 'html.parser')
                        deeper_links = [a['value'] for a in soup2.select("option") if 'value' in a.attrs]
                        #print(deeper_links)

                        for d in deeper_links:
                            if "milesplit" in d:
                                #print("- Scraping:", d)
                                try:
                                    d_resp = requests.get(d, allow_redirects=True)
                                    d_final_url = d_resp.url

                                    if "raw" in d_final_url:
                                        #print("------", d_final_url)
                                        raw.append(d_final_url)
                                except Exception as e:
                                    print(f"Error in second-level follow {d}: {e}")
                    except Exception as e:
                        print(f"Failed to parse second-level page: {e}")

            except Exception as e:
                print(f"Error following {h}: {e}")
    if raw:
        raw_links[l] = list(set(raw))
        print("---", len(raw_links[l]))
    else:
        raw_links[l] = [l]
        print("---", "NONE, Setting link to original.")

In [ ]:
for r in raw_links:
    print(r)
    for rl in raw_links[r]:
        print("|-----", rl)
        response = requests.get(rl)
        response.raise_for_status()  # Raise an exception for HTTP errors

        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        # Find the <pre> tag that contains the raw results
        pre_tag = soup.find('pre')
        if pre_tag:
            raw_results = pre_tag.get_text()
            print(raw_results)
        else:
            print("No raw results found on the page.")
        print("-----|")

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Step 1: Main meet results page
base_url = "https://wa.milesplit.com"
meet_results_url = "https://wa.milesplit.com/meets/15056-uw-indoor-invitational-2006/results"

# Step 2: Fetch and parse the main results page
response = requests.get(meet_results_url)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')

# Step 3: Find the raw results link
raw_link = None
for a_tag in soup.find_all("a", href=True):
    href = a_tag['href']
    if "/results/" in href:
        raw_link = urljoin(base_url, href)
        break

# Step 4: Output or fetch raw results
if raw_link:
    response = requests.get(url)
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    # Find the <pre> tag that contains the raw results
    pre_tag = soup.find('pre')
    if pre_tag:
        raw_results = pre_tag.get_text()
        print(raw_results)
    else:
        print("No raw results found on the page.")
else:
    print("No raw results link found.")